# Benchmarks orchestration

This notebook orchestrates the Benchmarks stage. It loads configuration, sets deterministic seeds, establishes the run context, and invokes `stages.benchmarks.run_benchmarks(cfg, ctx)`. All persistence is performed via the RunContext (metrics/tables/figures/reports/logs).

In [ ]:
import sys, pathlib
try:
    ROOT = pathlib.Path(__file__).resolve().parents[1]
except NameError:
    ROOT = pathlib.Path.cwd().resolve()
    if (ROOT / "utils").exists():
        pass
    elif (ROOT.parent / "utils").exists():
        ROOT = ROOT.parent
sys.path.append(str(ROOT))
print(f"Project root: {ROOT}")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import utils.config as ucfg
from utils.run import RunContext
from utils.plotting import set_matplotlib_style, place_legend_below
from utils import seeds as useeds
from stages.benchmarks import run_benchmarks

# Matplotlib policy
set_matplotlib_style()

In [ ]:
# Load and validate configuration
config_paths = [
    ROOT / "configs" / "default.yaml",
    ROOT / "configs" / "benchmarks.yaml",
]

try:
    cfg = ucfg.load_and_validate(config_paths)
except Exception:
    try:
        cfg = ucfg.load_config(config_paths)
    except Exception:
        cfg = ucfg.load(config_paths)

stage_name = "benchmarks"
seed = int(cfg.get("run", {}).get("seed", 42))
useeds.set_global_seeds(seed)
print(f"Loaded config for stage={stage_name}; seed={seed}")

In [ ]:
# Start run context and stage
run = RunContext.start(cfg)
ctx = run.stage(stage_name)

# Execute stage
_ = run_benchmarks(cfg, ctx)

# Close context (persist manifest) and run
inputs_manifest = {"configs": [str(p) for p in config_paths]}
try:
    ctx.close(inputs=inputs_manifest, notes="Benchmarks stage orchestrated via notebook.")
except Exception:
    pass
try:
    run.close()
except Exception:
    pass

print("Benchmarks stage completed.")